# 01 — Split Controls: leakage or covariate shift?

**This is the experiment that decides whether the paper is publishable.** Everything else is
supporting evidence.

Three strategies, identical ratios (70/20/10), identical partition sizes, 5 seeds, 2 models:

| strategy | grouping enforced | test position |
|---|---|---|
| `random_image` | no | random |
| `block_contiguous` | yes | tail of class |
| `block_shuffled` | yes | random |

**Decision rule**

* `block_shuffled ≈ block_contiguous ≪ random_image` → the drop is **grouping** → leakage. Paper stands.
* `block_shuffled ≈ random_image ≫ block_contiguous` → the drop is **position** → covariate shift. The leakage interpretation is wrong and must be retracted.
* in between → report both, quantify the share attributable to each.

Budget: ~9 GPU-h. Run CCAL overnight, EfficientNet-B0 is much faster.

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
SEEDS = [42, 123, 2024, 7, 2718]
STRATS = ["random_image", "block_contiguous", "block_shuffled"]
MODELS = ["effnet_b0", "ccal"]      # effnet first: fast, gives an early read
OUT = f"{RESULTS}/01_split_controls"; os.makedirs(OUT, exist_ok=True)

done = {os.path.basename(f).replace("_result.json", "")
        for f in os.listdir(OUT) if f.endswith("_result.json")} if os.path.isdir(OUT) else set()
print("already complete:", len(done))

In [ ]:
# Resumable: Kaggle sessions time out at 9h/12h, so skip finished cells.
results = []
t_start = time.time()
for model in MODELS:
    for strat in STRATS:
        for seed in SEEDS:
            name = f"{model}__{strat}__seed{seed}"
            if name in done:
                print("skip", name); continue
            r = engine.run_experiment(ROOT, model, strat, seed, OUT)
            results.append(r)
            print(f"    elapsed {(time.time()-t_start)/3600:.2f} h")

## Analysis — the decision

In [ ]:
all_res = engine.collect_results(OUT)
by = {}
for r in all_res:
    by.setdefault((r["model"], r["strategy"]), []).append(r)

print(f"{'model':<12}{'strategy':<20}{'acc mean':>10}{'sd':>7}{'BenignF1':>10}{'sd':>7}{'n':>4}")
import statistics as st
for (m, s), rs in sorted(by.items()):
    a = [x["accuracy"] for x in rs]; b = [x["benign_f1"] for x in rs]
    sd = lambda v: st.stdev(v) if len(v) > 1 else 0.0
    print(f"{m:<12}{s:<20}{st.mean(a):>10.2f}{sd(a):>7.2f}{st.mean(b):>10.2f}{sd(b):>7.2f}{len(a):>4}")

In [ ]:
print("Paired deltas vs random_image (bootstrap 95% CI over seeds)\n")
verdicts = {}
for m in MODELS:
    base = by.get((m, "random_image"), [])
    if not base: continue
    for s in ["block_contiguous", "block_shuffled"]:
        cur = by.get((m, s), [])
        if not cur: continue
        for key in ["accuracy", "benign_f1"]:
            d = metrics.paired_delta_ci(base, cur, key=key)
            print(f"{m:<11} {s:<18} {key:<10} delta {d['delta']:+7.2f}  "
                  f"CI [{d['lo']:+6.2f}, {d['hi']:+6.2f}]  excl0={d['excludes_zero']}")
            verdicts[(m, s, key)] = d
    print()

In [ ]:
# Explicit decision rule -- do not eyeball this.
for m in MODELS:
    if (m, "random_image") not in by: continue
    d_shuf = verdicts.get((m, "block_shuffled", "accuracy"))
    d_cont = verdicts.get((m, "block_contiguous", "accuracy"))
    if not (d_shuf and d_cont): continue
    grouping = -d_shuf["delta"]                    # attributable to grouping
    position = -d_cont["delta"] - grouping          # extra from tail-position
    print(f"{m}:")
    print(f"   total drop (contiguous)   {-d_cont['delta']:6.2f} pp")
    print(f"   attributable to GROUPING  {grouping:6.2f} pp   (block_shuffled vs random)")
    print(f"   attributable to POSITION  {position:6.2f} pp   (residual)")
    if d_shuf["excludes_zero"] and grouping > 0:
        print("   -> grouping effect is real; leakage interpretation SUPPORTED")
    else:
        print("   -> grouping effect not established; DO NOT claim leakage")
    print()

In [ ]:
with open(f"{RESULTS}/01_summary.json", "w") as f:
    json.dump({f"{k[0]}|{k[1]}|{k[2]}": v for k, v in verdicts.items()}, f, indent=2)
print("saved")